# Controlling MODFLOW 6 with the API

The MODFLOW 6 application programming interface (API) lets you drive a simulation from Python instead of running the `mf6` executable as a black box. Using the [`modflowapi`](https://github.com/MODFLOW-ORG/modflowapi) wrapper around the compiled MODFLOW 6 shared library, you can step through a model one time step (or even one solver iteration) at a time, read and modify internal variables while the model is running, and react to the simulated state.

This notebook works through a series of increasingly complex examples: running a model through the API, running a coupled flow-and-transport model, monitoring solver convergence, changing inputs on the fly with a callback, and finally implementing a streamflow-augmentation rule that turns a well on only when simulated streamflow drops too low.

## Imports and setup

Import FloPy, the `modflowapi` interface, and the supporting plotting and data libraries, then create a directory for the figures saved by this notebook.

In [ ]:
%matplotlib inline
import flopy
import matplotlib.pyplot as plt
from modflowapi import ModflowApi, Callbacks, run_simulation
import numpy as np
import os
import pandas as pd
import pathlib as pl
import platform

In [ ]:
fig_path = pl.Path(".figures")
fig_path.mkdir(exist_ok=True)

In [ ]:
sample_frequency = "annual"  # monthly or annual
name = "sv"

### Locate the MODFLOW 6 library

The API drives MODFLOW 6 through its compiled shared library (`libmf6`), not the `mf6` command-line executable. Find the library (`.so` on Linux, `.dylib` on macOS, `.dll` on Windows) and the executable inside the active Conda environment, and confirm that both exist.

In [ ]:
env_path = pl.Path(os.environ.get("CONDA_PREFIX", None))
assert env_path is not None, "Notebook must be run from the mf6xtd pixi environment"

bin_path = "bin"
exe_ext = ""
if "linux" in platform.platform().lower():
    lib_ext = ".so"
elif "darwin" in platform.platform().lower() or "macos" in platform.platform().lower():
    lib_ext = ".dylib"
else:
    bin_path = "Scripts"
    lib_ext = ".dll"
    exe_ext = ".exe"
lib_name = env_path / f"{bin_path}/libmf6{lib_ext}"
if not lib_name.is_file():
    lib_name = env_path / f"lib/libmf6{lib_ext}"
if not lib_name.is_file():
    raise FileNotFoundError(
        f"Could not find mf6 library at in either: {env_path / 'bin'}"
        + f" or {env_path / 'lib'}"
    )

mf6_exe = env_path / f"{bin_path}/mf6{exe_ext}"

In [ ]:
lib_name.is_file(), mf6_exe.is_file()

## Run a model with the API

The simplest use of the API: load an existing model, then drive it through time from Python. Instead of calling the `mf6` executable, initialize the library, repeatedly call `update()` to advance one time step until the end of the simulation is reached, and then finalize. This produces the same results as running the model with the executable.

In [ ]:
ws = pl.Path(
    f"../data/synthetic-valley/synthetic-valley-base-{sample_frequency}"
).resolve()
new_ws = pl.Path(f"models/synthetic-valley-base-{sample_frequency}_api01")

sim = flopy.mf6.MFSimulation.load(sim_name=name, sim_ws=ws, write_headers=False)
sim.set_sim_path(new_ws)
sim.write_simulation()


In [ ]:
mf6 = ModflowApi(lib_name, working_directory=new_ws)
mf6.initialize()

current_time = mf6.get_current_time()
end_time = mf6.get_end_time()

while current_time < end_time:
    print(f"Running simulation time: {current_time}")
    mf6.update()
    current_time = mf6.get_current_time()
mf6.finalize()

## Run a model with a manual solver loop

This example uses the watershed groundwater flow (GWF) model to show the manual solver loop. Instead of calling `update()`, it prepares each time step and then iterates the solver itself (`prepare_solve` / `solve` / `finalize_solve`) until the solution converges. Working at this level lets you read or modify the solution between outer iterations.

In [ ]:
ws = pl.Path(f"../data/watershed/gwf-only")
new_ws = pl.Path(f"models/watershed-gwf_api02")

sim = flopy.mf6.MFSimulation.load(sim_name=name, sim_ws=ws, write_headers=False)
sim.set_sim_path(new_ws)
sim.write_simulation()

In [ ]:
mf6 = ModflowApi(lib_name, working_directory=new_ws)
mf6.initialize()

# maximum outer iterations
mxit_tag = mf6.get_var_address("MXITER", "SLN_1")
max_iter = mf6.get_value(mxit_tag)

current_time = mf6.get_current_time()
end_time = mf6.get_end_time()

while current_time < end_time:
    print(f"Running simulation time: {current_time}")

    # get dt and prepare for non-linear iterations
    dt = mf6.get_time_step()
    mf6.prepare_time_step(dt)

    # convergence loop
    kiter = 0
    mf6.prepare_solve()

    while kiter < max_iter:
        # here is where you could update well rates or other time-varying
        # inputs that depend on the solution from the previous iteration.
        # For example, you could get the current head solution and use it to
        # update a well rate based on a specified head difference and conductance.

        # solve with updated well rate
        has_converged = mf6.solve()
        kiter += 1

        if has_converged:
            msg = f"  Component {1}" + f" converged in {kiter}" + " outer iterations"
            print(msg)
            break

    if not has_converged:
        print("model did not converge")

    # finalize time step
    mf6.finalize_solve()

    # finalize time step and update time
    mf6.finalize_time_step()
    current_time = mf6.get_current_time()

print("Finalize simulation.")
mf6.finalize()

## Run a coupled flow and transport model

Extend the manual solver loop to a simulation with more than one solution. The watershed model couples a groundwater flow (GWF) and a transport (GWT) model, so each time step loops over both solutions (`SLN_1` and `SLN_2`), solving each to convergence in turn.

In [ ]:
ws = pl.Path(f"../data/watershed/gwf-gwt")
new_ws = pl.Path(f"models/watershed-gwf-gwt_api03")

sim = flopy.mf6.MFSimulation.load(sim_name=name, sim_ws=ws, write_headers=False)
sim.set_sim_path(new_ws)
sim.write_simulation()

In [ ]:
mf6 = ModflowApi(lib_name, working_directory=new_ws)
mf6.initialize()

# maximum outer iterations
mxit_tag = mf6.get_var_address("MXITER", "SLN_1")
max_iter_gwf = mf6.get_value(mxit_tag)
mxit_tag = mf6.get_var_address("MXITER", "SLN_2")
max_iter_gwt = mf6.get_value(mxit_tag)


current_time = mf6.get_current_time()
end_time = mf6.get_end_time()

while current_time < end_time:
    print(f"Running simulation time: {current_time}")

    # get dt and prepare for non-linear iterations
    dt = mf6.get_time_step()
    mf6.prepare_time_step(dt)

    # convergence loop
    for km in (1, 2):  # loop over gwf and gwt
        kiter = 0
        max_iter = max_iter_gwf if km == 1 else max_iter_gwt
        mf6.prepare_solve(component_id=km)

        while kiter < max_iter:
            # here is where you could update well rates or other time-varying
            # inputs that depend on the solution from the previous iteration.
            # For example, you could get the current head solution and use it to
            # update a well rate based on a specified head difference and conductance.
            if km == 1:
                pass  # update gwf inputs if needed
            else:
                pass  # update gwt inputs if needed

            # solve with updated well rate
            has_converged = mf6.solve(component_id=km)
            kiter += 1

            if has_converged:
                msg = (
                    f"  Component {km}" + f" converged in {kiter}" + " outer iterations"
                )
                print(msg)
                break

        if not has_converged:
            print("model did not converge")

        # finalize time step
        mf6.finalize_solve(component_id=km)

    # finalize time step and update time
    mf6.finalize_time_step()
    current_time = mf6.get_current_time()

print("Finalize simulation.")
mf6.finalize()

## Monitor convergence

Step into the solver to watch how the solution converges. After preparing each time step, loop over the solver's outer iterations manually and read the maximum head change (`HNCG`) at every iteration. The `|maximum head change|` for every iteration is plotted on a log scale and the figure is refreshed after each time step, so you can watch convergence happen **as the model runs**. The x-axis is grouped by stress period and time step (read from the `TDIS` package, with labels thinned when there are many), and the figure uses the USGS publication style from `flopy.plot.styles`. This exposes the inner workings that are normally hidden when MODFLOW 6 runs on its own, and is useful for diagnosing slow or failing convergence.

In [ ]:
ws = pl.Path(
    f"../data/synthetic-valley/synthetic-valley-base-{sample_frequency}"
)
new_ws = pl.Path(f"models/synthetic-valley-base-{sample_frequency}_api04")

sim = flopy.mf6.MFSimulation.load(sim_name=name, sim_ws=ws, write_headers=False)
sim.memory_print_option = "all"
sim.set_sim_path(new_ws)
sim.write_simulation()

In [ ]:
from IPython.display import clear_output, display

mf6 = ModflowApi(lib_name, working_directory=new_ws)
mf6.initialize()

# solver settings and the live pointer to the maximum head change per outer
# iteration (HNCG); KPER and KSTP are read from the TDIS package each time step
max_iter = int(mf6.get_value(mf6.get_var_address("MXITER", "SLN_1"))[0])
max_change = mf6.get_value_ptr(mf6.get_var_address("HNCG", "SLN_1"))
kper_tag = mf6.get_var_address("KPER", "TDIS")
kstp_tag = mf6.get_var_address("KSTP", "TDIS")

current_time = mf6.get_current_time()
end_time = mf6.get_end_time()

# convergence history, one entry per outer iteration:
# (cumulative iteration, |maximum head change|, stress period, time step)
history = []
not_converged = []
max_ticks = 25  # most SP/TS labels to show before thinning


def plot_convergence(fig, ax):
    ax.clear()
    if history:
        y = np.array([h[1] for h in history], dtype=float)
        y[y == 0.0] = np.nan  # skip exact zeros on the log axis
        ax.semilogy(
            [h[0] for h in history],
            y,
            marker="o",
            ms=4,
            lw=1.0,
            color="0.25",
            mfc="cyan",
            mec="0.25",
        )

        # group consecutive iterations into (stress period, time step) blocks
        blocks, b0 = [], 0
        for k in range(len(history)):
            if k == len(history) - 1 or history[k][2:] != history[k + 1][2:]:
                blocks.append((b0, k))
                b0 = k + 1

        # thin the labels/separators so at most ~max_ticks are drawn
        stride = max(1, -(-len(blocks) // max_ticks))
        ticks, labels = [], []
        for bi, (i0, i1) in enumerate(blocks):
            if bi % stride == 0 or bi == len(blocks) - 1:
                ticks.append(0.5 * (history[i0][0] + history[i1][0]))
                labels.append(f"SP {history[i1][2]}\nTS {history[i1][3]}")
                if i0 > 0:
                    ax.axvline(history[i0][0] - 0.5, color="0.5", lw=0.5)
        ax.set_xticks(ticks)
        ax.set_xticklabels(labels)

    flopy.plot.styles.heading(ax=ax, heading="Outer iteration convergence")
    flopy.plot.styles.xlabel(ax=ax, label="Stress period and time step")
    flopy.plot.styles.ylabel(ax=ax, label="Maximum head change, ft")
    flopy.plot.styles.remove_edge_ticks(ax=ax)

    clear_output(wait=True)
    display(fig)


with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)

    niter = 0
    while current_time < end_time:
        dt = mf6.get_time_step()
        mf6.prepare_time_step(dt)
        kper = int(mf6.get_value(kper_tag)[0])
        kstp = int(mf6.get_value(kstp_tag)[0])

        mf6.prepare_solve()
        kiter = 0
        has_converged = False
        while kiter < max_iter:
            has_converged = mf6.solve()
            niter += 1
            history.append((niter, abs(float(max_change[kiter])), kper, kstp))
            kiter += 1
            if has_converged:
                break

        plot_convergence(fig, ax)  # live update once per time step

        if not has_converged:
            not_converged.append((kper, kstp))

        mf6.finalize_solve()
        mf6.finalize_time_step()
        current_time = mf6.get_current_time()

    mf6.finalize()

plt.close(fig)  # the live figure is already displayed; avoid a duplicate

if not_converged:
    print("did not converge:", ", ".join(f"SP {p} TS {t}" for p, t in not_converged))
else:
    print("all time steps converged")

## Increase precipitation with a callback

Modify model inputs while the simulation is running. The `callback_function` below is called by `modflowapi` at key points in the solution (for example `Callbacks.initialize`, `Callbacks.stress_period_start`, and `Callbacks.iteration_start`); here it intercepts the start of each stress period and multiplies the recharge by 1.1, increasing precipitation on the fly without rewriting the input files.

The callback is the most convenient way to work with the standard stress packages. `modflowapi` exposes the running model as familiar Python objects - the simulation (`sim`), each model (`sim.sv`), and its packages (`ml.rch_0`) - so you can reach a package by name and read or change its `stress_period_data` directly with numpy-style syntax (`spd["recharge"] *= 1.1`). The edits take effect in the running model immediately, and you never have to know the internal MODFLOW variable names, memory addresses, or array layout.

For more complicated use of the MODFLOW API you may need lower-level access to data that is not exposed through the package interface - for example internal solver variables or advanced-package terms. In those cases you read a variable directly from the model's memory as a copy with `get_value` (a snapshot you can inspect) or as a pointer with `get_value_ptr` (a live view that stays in sync with the running model and can be written back). The streamflow-augmentation example below uses this lower-level access to read the simulated SFR flow and set the prediction-well rate.

In [ ]:
ws = pl.Path(
    f"../data/synthetic-valley/synthetic-valley-base-{sample_frequency}"
)
new_ws = pl.Path(f"models/synthetic-valley-base-{sample_frequency}_api04")

sim = flopy.mf6.MFSimulation.load(sim_name=name, sim_ws=ws, write_headers=False)
sim.set_sim_path(new_ws)
sim.write_simulation()

In [ ]:
def callback_function(sim, callback_step):
    """
    A demonstration function that dynamically adjusts recharge in a
    MODFLOW 6 model through the MODFLOW-API

    Parameters
    ----------
    sim : modflowapi.ApiSimulation
        A simulation object for the solution group that is
        currently being solved
    step : enumeration
        modflowapi.Callbacks enumeration object that indicates
        the part of the solution modflow is currently in.
    """
    ml = sim.sv
    if callback_step == Callbacks.initialize:
        print(sim.models)

    if callback_step == Callbacks.stress_period_start:
        rcha = ml.rch_0
        spd = rcha.stress_period_data
        print(f"updating recharge: stress_period={ml.kper + 1}")
        print(spd)
        spd["recharge"] *= 1.1

    if callback_step == Callbacks.timestep_start:
        pass

    if callback_step == Callbacks.iteration_start:
        # we can implement complex solutions to boundary conditions here!
        pass

In [ ]:
run_simulation(lib_name, new_ws, callback_function, verbose=False)

## Augment streamflow with the prediction well

A more realistic application: use the API to implement a real-time operating rule. The goal is to keep simulated flow at the southern stream gage above a minimum threshold by pumping a prediction well only when needed. We first simulate a baseline scenario with the well turned off, then re-run with a callback that switches the well on whenever streamflow drops below the minimum, and finally compare the two.

### Simulate the baseline scenario

Run the advanced model with the prediction well turned off to establish the un-augmented streamflow at the southern gage. The head map shows the model boundaries (stream, pumping wells, and prediction well), and the streamflow is collected into a dataframe for later comparison.

In [ ]:
# run the base model
start_date = pd.to_datetime("1962-01-01 00:00:00")


In [ ]:
ws = pl.Path(
    f"models/synthetic-valley-advanced-{sample_frequency}"
)
new_ws = pl.Path(f"models/synthetic-valley-advanced-{sample_frequency}_api05")


sim = flopy.mf6.MFSimulation.load(
    sim_name=name, sim_ws=ws, exe_name=mf6_exe, write_headers=False
)
gwf = sim.get_model("sv")
pak = gwf.get_package("prediction")
new_spd = {}
for kper in range(sim.tdis.nper.array):
    spd = pak.stress_period_data.get_data(kper)
    if spd is None:
        continue
    spd["q"] = 0.0
    new_spd[kper] = spd
pak.stress_period_data.set_data(new_spd)

sim.set_sim_path(new_ws)
sim.write_simulation(silent=True)
sim.run_simulation(silent=True)

In [ ]:
aspect_ratio = 25.0 / 40.0
height = 6.6
width = aspect_ratio * height
with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(
        1,
        1,
        figsize=(width, height),
        sharex=True,
        constrained_layout=True,
    )
    mv = flopy.plot.PlotMapView(model=gwf, layer=0, ax=ax, extent=gwf.modelgrid.extent)
    hds = gwf.output.head().get_data()
    hds[hds == 1e30] = 14.066030871906671
    mv.plot_array(hds)
    CS = mv.contour_array(hds, levels=np.arange(1, 14, 1), colors="black")
    ax.clabel(CS, CS.levels, fmt="%.0f", fontsize=10)
    mv.plot_bc("SFR", color="green")
    mv.plot_bc(
        "WEL",
        package=gwf.pwell,
        plotAll=True,
        color="red",
        kper=12,
    )
    mv.plot_bc("WEL", package=gwf.prediction, plotAll=True, color="orange", kper=12)
    mv.plot_grid(lw=0.5, color="black")

    ax.set_xticklabels([])
    ax.set_yticklabels([])

    fig.savefig(fig_path / "sv-head.png", dpi=300, transparent=False)

In [ ]:
base_df = gwf.sfr.output.obs().get_dataframe(start_datetime=start_date)
base_df["RIV-FLOW"] /= -86400
base_df["RIV-SWGW"] /= -86400
base_df["TOTAL"] = base_df["RIV-SWGW"]

Q0 = base_df["TOTAL"].iloc[0]
base_df["PCT_DIFF"] = -100.0 * (base_df["TOTAL"] - Q0) / Q0


In [ ]:
new_row = {
    start_date: {
        "totim": 0.0,
        "RIV-FLOW": np.nan,
        "RIV-SWGW": np.nan,
        "TOTAL": np.nan,
        "PCT_DIFF": np.nan,
        "augmentation rate": np.nan,
    }
}
new_row = pd.DataFrame(new_row).T
base_df = pd.concat([new_row, base_df])
base_df

### Define the streamflow-augmentation rule

Set the minimum acceptable streamflow and write the callback that enforces it. When the downstream SFR flow falls below `min_flow`, the prediction well is turned on at a rate proportional to the shortfall (capped at a maximum), and that water is added as stream inflow.

The callback always sets the rate at the start of each stress period using the streamflow from the end of the previous timestep. Setting `update_each_iteration = True` additionally recomputes the rate at the start of every outer iteration (`Callbacks.iteration_start`), so the augmentation tracks the latest simulated streamflow and converges with the rest of the solution instead of lagging it by one timestep.

To keep the iterations stable, the rate is *ratcheted* within each timestep: once the pump turns on in a Picard iteration it stays on, and the rate can only hold steady or increase (never drop back to zero) in later iterations of that timestep. The pump can still turn off at the start of a later timestep when streamflow recovers.

In [ ]:
min_flow = 15.0

# when True, the augmentation rate is also recomputed every outer iteration
# (Callbacks.iteration_start) from the latest streamflow; when False, it is set
# once per stress period from the previous timestep's streamflow
update_each_iteration = True

In [ ]:
# run the advanced model in the augmentation workspace through the API with the
# streamflow-augmentation callback defined below (ws0 is populated separately)
sim = flopy.mf6.MFSimulation.load(sim_name=name, sim_ws=new_ws, write_headers=False)

In [ ]:
# aug_floor is the lowest augmentation rate allowed during the current timestep's
# Picard iterations; it is reset to 0 at the start of each timestep so the pump can
# turn off between timesteps but never within a timestep's iterations
aug_floor = 0.0


def callback_function(sim, callback_step):
    """
    A demonstration callback that augments streamflow by pumping the
    prediction well whenever simulated flow at the southern SFR gage
    drops below a minimum threshold (min_flow).

    The rate is always set at the start of each stress period using the
    streamflow from the end of the previous timestep. When
    update_each_iteration is True, the rate is additionally recomputed at the
    start of every outer iteration so that it tracks the latest simulated
    streamflow within the timestep instead of lagging it by one timestep.

    Within a timestep the rate is ratcheted: once the pump turns on in a Picard
    iteration it cannot turn off in a later iteration - it can only stay the same
    or increase. This avoids on/off oscillation between iterations. The pump can
    still turn off at the start of a later timestep when streamflow recovers.

    Parameters
    ----------
    sim : modflowapi.ApiSimulation
        A simulation object for the solution group that is
        currently being solved
    step : enumeration
        modflowapi.Callbacks enumeration object that indicates
        the part of the solution modflow is currently in.
    """
    global aug_floor

    if sample_frequency == "monthly":
        aug_start = 120
    else:
        aug_start = 10

    ml = sim.sv

    def update_augmentation():
        """Read the latest downstream SFR flow and set the prediction-well
        pumping rate (and matching stream inflow) needed to keep flow at or
        above min_flow. The rate can only increase within a timestep (see
        aug_floor), so the pump never switches back off between iterations."""
        global aug_floor
        tag = sim.mf6.get_var_address("DSFLOW", "SV", "SFR-1")
        arr = sim.mf6.get_value(tag)
        arr /= 86400.0
        tag = sim.mf6.get_var_address("INFLOW", "SV", "SFR-1")
        inflow = sim.mf6.get_value_ptr(tag)
        q = float(arr[-1])
        if q < min_flow:
            rate = min(2.5 * (min_flow - q) * 86400.0, 2e6)
        else:
            rate = 0.0
        # never reduce the rate within a timestep: once on, stay on
        rate = max(rate, aug_floor)
        aug_floor = rate
        ml.prediction.stress_period_data["q"] = -rate
        inflow[0] = rate

    if callback_step == Callbacks.initialize:
        print(sim.models)

    if callback_step == Callbacks.stress_period_start:
        # start each stress period with the pump allowed to be off
        aug_floor = 0.0
        if sim.kper > aug_start:
            update_augmentation()

    if callback_step == Callbacks.timestep_start:
        # allow the pump to start from off at the beginning of each timestep
        aug_floor = 0.0

    if callback_step == Callbacks.iteration_start:
        # recompute the augmentation rate from the latest streamflow so it
        # converges with the rest of the solution instead of lagging a timestep
        if update_each_iteration and sim.kper > aug_start:
            update_augmentation()

In [ ]:
run_simulation(lib_name, new_ws, callback_function, verbose=False)

### Compare baseline and augmented streamflow

Collect the augmented streamflow and the well augmentation rate from the model output, then plot the baseline and augmented flows together with the augmentation period and the minimum-flow threshold to see how the operating rule keeps streamflow above the target.

In [ ]:
gwf = sim.get_model("SV")
df = gwf.sfr.output.obs().get_dataframe(start_datetime=start_date)
df["RIV-FLOW"] /= -86400
df["RIV-SWGW"] /= -86400
df["TOTAL"] = df["RIV-SWGW"]


Q0 = df["TOTAL"].iloc[0]
df["PCT_DIFF"] = -100.0 * (df["TOTAL"] - Q0) / Q0
df

In [ ]:
cobj = gwf.output.budget()
cobj.headers.paknam2.drop_duplicates()
prediction_rate = []
for totim in cobj.get_times():
    rate = cobj.get_data(totim=totim, paknam2="prediction")[0]
    if len(rate) == 0:
        q = 0.0
    else:
        q = float(rate["q"][0])
    prediction_rate.append(q)
prediction_rate = np.array(prediction_rate) / -86400.0
prediction_rate[prediction_rate == 0.0] = np.nan

In [ ]:
df["augmentation rate"] = prediction_rate

In [ ]:
new_row = {
    start_date: {
        "totim": 0.0,
        "RIV-FLOW": np.nan,
        "RIV-SWGW": np.nan,
        "TOTAL": np.nan,
        "PCT_DIFF": np.nan,
        "augmentation rate": np.nan,
    }
}
new_row = pd.DataFrame(new_row).T
new_row

In [ ]:
df = pd.concat([new_row, df])
df

In [ ]:
if sample_frequency == "monthly":
    aug_start = 120
else:
    aug_start = 10
x0 = df.index[aug_start]
x1 = df.index[-1]
xstart = df.index[1]
xend = df.index[-1]

In [ ]:
base_color, aug_color = "green", "blue"
drawstyle = "steps-pre"

with flopy.plot.styles.USGSPlot():
    fig, axs = plt.subplots(
        2,
        1,
        figsize=(12.3, 4.3),
        sharex=True,
        constrained_layout=True,
    )

    fig.suptitle("Southern Boundary - Gage 1")

    ax = axs[0]
    y0, y1 = 5, 30
    base_df["RIV-FLOW"][xstart:xend].plot(
        ax=ax,
        color=base_color,
        lw=1.5,
        ls="-",
        # clip_on=False,
        drawstyle=drawstyle,
        label="Base",
    )
    df["RIV-FLOW"][xstart:xend].plot(
        ax=ax,
        color=aug_color,
        lw=1.5,
        ls="-",
        # clip_on=False,
        drawstyle=drawstyle,
        label="Augmented flow",
    )
    ax.axhline(0, lw=0.5, color="black")
    ax.axhline(min_flow, lw=1.0, ls=":", color="blue", label="Minimum flow")
    ax.fill_between(
        [x0, x1],
        y0,
        y1,
        color="cyan",
        alpha=0.25,
        lw=0,
        label="Augmentation period",
    )

    # ax.set_xlim(xstart, xend)
    ax.set_ylim(y0, y1)

    ax.set_ylabel("River\nDischarge, cfs")
    leg = ax.legend(loc="upper left", ncol=2, frameon=False)

    ax = axs[1]
    y0, y1 = 0, 25

    area_df = df["augmentation rate"][xstart:xend].copy()

    area_df.plot(
        ax=ax,
        color="red",
        lw=1.5,
        ls="-",
        # clip_on=False,
        drawstyle=drawstyle,
        label="Augmented flow",
    )
    ax.fill_between(
        [x0, x1],
        y0,
        y1,
        color="cyan",
        alpha=0.25,
        lw=0,
        label="Augmentation period",
    )

    for idx in range(1, len(df.index)):
        xa0, xa1 = df.index[idx - 1], df.index[idx]
        ax.fill_between(
            [xa0, xa1],
            0,
            df["augmentation rate"].values[idx],
            color="red",
            lw=1.0,
            edgecolor="black",
            zorder=100,
        )
    # ax.set_xlim(xstart, xend)
    ax.set_ylim(y0, y1)

    ax.set_ylabel("Augmentation\nrate, cfs")
    ax.set_xlabel("Year")
    # leg = ax.legend(loc="upper left", ncol=2, frameon=False)

    fig.align_ylabels()
    fig.savefig(
        fig_path / "sv_sfr_api_southern_boundary.png", dpi=300, transparent=False
    )

### Compare the two update strategies

Run the augmentation model twice - once recomputing the rate only at the start of each stress period (`update_each_iteration = False`) and once recomputing it every outer iteration (`update_each_iteration = True`) - and store each result in its own dataframe (`df_period` and `df_iter`). Plotting them together with the baseline shows how updating the rate within the timestep keeps the southern-gage flow closer to the `min_flow` target.

In [ ]:
# run the augmentation model with both update strategies, saving each run to its
# own dataframe (the baseline run is already stored in base_df)
results = {}
for label, flag in (("per stress period", False), ("per iteration", True)):
    update_each_iteration = flag
    run_simulation(lib_name, new_ws, callback_function, verbose=False)

    gwf = sim.get_model("SV")
    rdf = gwf.sfr.output.obs().get_dataframe(start_datetime=start_date)
    rdf["RIV-FLOW"] /= -86400
    rdf["RIV-SWGW"] /= -86400
    rdf["TOTAL"] = rdf["RIV-SWGW"]
    Q0 = rdf["TOTAL"].iloc[0]
    rdf["PCT_DIFF"] = -100.0 * (rdf["TOTAL"] - Q0) / Q0

    # prediction-well (augmentation) rate from the cell-by-cell budget
    cobj = gwf.output.budget()
    prediction_rate = []
    for totim in cobj.get_times():
        rate = cobj.get_data(totim=totim, paknam2="prediction")[0]
        q = 0.0 if len(rate) == 0 else float(rate["q"][0])
        prediction_rate.append(q)
    prediction_rate = np.array(prediction_rate) / -86400.0
    prediction_rate[prediction_rate == 0.0] = np.nan
    rdf["augmentation rate"] = prediction_rate

    # prepend a start-date row so the series begins at the simulation start
    start_row = pd.DataFrame({start_date: {c: np.nan for c in rdf.columns}}).T
    start_row["totim"] = 0.0
    results[label] = pd.concat([start_row, rdf])

# the two strategies are kept in separate dataframes
df_period = results["per stress period"]
df_iter = results["per iteration"]
df_iter

In [ ]:
if sample_frequency == "monthly":
    aug_start = 120
else:
    aug_start = 10
idx_ref = df_iter.index
x0, x1 = idx_ref[aug_start], idx_ref[-1]
xstart, xend = idx_ref[1], idx_ref[-1]
drawstyle = "steps-pre"

with flopy.plot.styles.USGSPlot():
    fig, axs = plt.subplots(
        2, 1, figsize=(12.3, 4.3), sharex=True, constrained_layout=True
    )
    fig.suptitle("Southern Boundary - Gage 1: update-strategy comparison")

    ax = axs[0]
    ax.set_ylim(5, 30)
    base_df["RIV-FLOW"][xstart:xend].plot(
        ax=ax, color="green", lw=1.5, drawstyle=drawstyle, label="Base"
    )
    df_period["RIV-FLOW"][xstart:xend].plot(
        ax=ax, color="orange", lw=1.5, drawstyle=drawstyle,
        label="Augmented (per stress period)",
    )
    df_iter["RIV-FLOW"][xstart:xend].plot(
        ax=ax, color="blue", lw=1.5, drawstyle=drawstyle,
        label="Augmented (per iteration)",
    )
    ax.axhline(min_flow, lw=1.0, ls=":", color="black", label="Minimum flow")
    ax.fill_between(
        [x0, x1], 5, 30, color="cyan", alpha=0.25, lw=0, label="Augmentation period"
    )
    ax.set_ylabel("River\nDischarge, cfs")
    ax.legend(loc="upper left", ncol=2, frameon=False)

    ax = axs[1]
    y0, y1 = 0, 25
    ax.set_ylim(y0, y1)
    df_period["augmentation rate"][xstart:xend].plot(
        ax=ax, color="orange", lw=1.5, drawstyle=drawstyle, label="per stress period"
    )
    df_iter["augmentation rate"][xstart:xend].plot(
        ax=ax, color="blue", lw=1.5, drawstyle=drawstyle, label="per iteration"
    )
    ax.fill_between([x0, x1], y0, y1, color="cyan", alpha=0.25, lw=0)
    ax.set_ylabel("Augmentation\nrate, cfs")
    ax.set_xlabel("Year")
    ax.legend(loc="upper left", ncol=2, frameon=False)

    fig.align_ylabels()
    fig.savefig(
        fig_path / "sv_sfr_api_update_comparison.png", dpi=300, transparent=False
    )

In [ ]:
# quick numeric summary over the (shaded) augmentation period
if sample_frequency == "monthly":
    aug_start = 120
else:
    aug_start = 10


def summarize(df, label):
    rate = df["augmentation rate"].fillna(0.0)  # cfs, 0 when the well is off
    dt_s = df["totim"].diff().fillna(0.0) * 86400.0  # period length, seconds
    aug = df.iloc[aug_start:]  # rows within the augmentation period
    flow = aug["RIV-FLOW"]
    return pd.Series(
        {
            "min gage flow (cfs)": flow.min(),
            "mean gage flow (cfs)": flow.mean(),
            "periods below target": int((flow < min_flow).sum()),
            "max augmentation (cfs)": rate.iloc[aug_start:].max(),
            "total augmented volume (acre-ft)": (rate * dt_s).iloc[aug_start:].sum()
            / 43560.0,
        },
        name=label,
    )


summary = pd.DataFrame(
    [
        summarize(base_df, "base"),
        summarize(df_period, "per stress period"),
        summarize(df_iter, "per iteration"),
    ]
)
summary.round(2)